<a href="https://colab.research.google.com/github/Datahuntl/Estudo-Comparativo-de-Detec-o-de-DeepFakes/blob/main/Processamento_do_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
import shutil
from google.colab import drive
import cv2

drive.mount("/content/drive")

In [ ]:
dataset_dir = "/content/drive/MyDrive/IC DeepFakes/DATASET"
json_file_path = os.path.join(dataset_dir, "metadata.json")  # substitua 'metadata.json' pelo nome real do seu arquivo

output_fake_dir = os.path.join(dataset_dir, "FAKE")
output_real_dir = os.path.join(dataset_dir, "REAL")

In [ ]:
os.makedirs(output_fake_dir, exist_ok=True)
os.makedirs(output_real_dir, exist_ok=True)


In [ ]:
print("Lendo o arquivo JSON...")
with open(json_file_path, "r") as f:
    metadata = json.load(f)

In [ ]:
print("Iniciando a separação dos vídeos...")
videos_movidos = 0

for video_name, info in metadata.items():
    video_path = os.path.join(dataset_dir, video_name)

    # Verifica se o arquivo de vídeo realmente existe na pasta antes de tentar mover
    if os.path.exists(video_path):
        label = info["label"]

        # Define o destino com base no label (FAKE ou REAL)
        if label == "FAKE":
            dest_path = os.path.join(output_fake_dir, video_name)
        elif label == "REAL":
            dest_path = os.path.join(output_real_dir, video_name)
        else:
            print(f"Label desconhecido para {video_name}: {label}")
            continue

        # Move o arquivo (mude para shutil.copy se preferir manter uma cópia na pasta original)
        shutil.move(video_path, dest_path)
        videos_movidos += 1
    else:
        # Ignora silenciosamente ou avisa caso o JSON mencione um vídeo que não foi baixado
        pass

print(
    f"Processo concluído! {videos_movidos} vídeos foram organizados com sucesso."
)

Iniciando a separação dos vídeos...
Processo concluído! 2473 vídeos foram organizados com sucesso.


In [ ]:
import cv2

PATH_REAL_VIDEOS = "/content/drive/MyDrive/IC DeepFakes/DATASET/ORIGINAL/REAL"
PATH_FAKE_VIDEOS = "/content/drive/MyDrive/IC DeepFakes/DATASET/ORIGINAL/FAKE"

DRIVE_REAL_FACES = "/content/drive/MyDrive/IC DeepFakes/DATASET_FACES/PROCESSADO/REAL"
DRIVE_FAKE_FACES = "/content/drive/MyDrive/IC DeepFakes/DATASET_FACES/PROCESSADO/FAKE"
os.makedirs(DRIVE_REAL_FACES, exist_ok=True)
os.makedirs(DRIVE_FAKE_FACES, exist_ok=True)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def extract_faces_permanently(video_dir, output_dir, max_faces_per_video=15):
    """
    Varre os vídeos salvos no Drive, extrai faces e salva permanentemente em subpastas no Drive.
    """
    video_files = [os.path.join(video_dir, f) for f in os.listdir(video_dir) if f.lower().endswith('.mp4')]
    print(f"Processando {len(video_files)} vídeos de: {os.path.basename(video_dir)}")

    face_count = 0
    # Como são muitos vídeos (2.473), vamos colocar um contador visual simples
    for idx, video_path in enumerate(video_files):
        if idx % 50 == 0:
            print(f" Progresso: [{idx}/{len(video_files)}] vídeos verificados...")

        cap = cv2.VideoCapture(video_path)
        faces_saved_from_this_video = 0
        frame_idx = 0

        while cap.isOpened() and faces_saved_from_this_video < max_faces_per_video:
            ret, frame = cap.read()
            if not ret:
                break

            # Captura 1 frame a cada 10 para varrer o vídeo todo sem repetir a mesma pose
            if frame_idx % 10 == 0:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

                for (x, y, w, h) in faces:
                    if faces_saved_from_this_video >= max_faces_per_video:
                        break

                    face_crop = frame[y:y+h, x:x+w]

                    # Nome único baseado no vídeo original e número sequencial
                    video_base_name = os.path.splitext(os.path.basename(video_path))[0]
                    img_name = f"face_{video_base_name}_{faces_saved_from_this_video}.jpg"

                    cv2.imwrite(os.path.join(output_dir, img_name), face_crop)
                    face_count += 1
                    faces_saved_from_this_video += 1

            frame_idx += 1
        cap.release()
    print(f"✨ Concluído! Total de faces extraídas e salvas no Drive: {face_count}\n")

# Executar a extração em massa
print("Iniciando extração em massa de vídeos REAL...")
extract_faces_permanently(PATH_REAL_VIDEOS, DRIVE_REAL_FACES)

print("Iniciando extração em massa de vídeos FAKE...")
extract_faces_permanently(PATH_FAKE_VIDEOS, DRIVE_FAKE_FACES)